# Stage 5 — Model Evaluation & Analysis

Tujuan: Mengevaluasi performa pipeline menggunakan Akurasi, Presisi, Recall, dan F1-Score.

# 5.1 Persiapan Data dan Kalkulasi Metrik
 
Memuat pustaka, membaca data hasil prediksi, dan mendefinisikan fungsi untuk menghitung metrik evaluasi klasifikasi untuk membandingkan performa pendekatan Cosine Similarity dan SVM.

In [1]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

PROJECT_ROOT = Path.cwd().parent
RESULTS_DIR = PROJECT_ROOT / 'data' / 'results'
EVAL_DIR = PROJECT_ROOT / 'data' / 'eval'

pred_df = pd.read_csv(RESULTS_DIR / 'predictions.csv')
y_actual = pred_df['actual_pasal']
y_cosine = pred_df['cosine_predicted_pasal']
y_svm = pred_df['svm_predicted_pasal']
all_labels = sorted(set(y_actual) | set(y_cosine) | set(y_svm))

In [2]:
def calc_metrics(y_true, y_pred, name):
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'Recall': recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'F1-score': f1_score(y_true, y_pred, average='weighted', zero_division=0)
    }

metrics = [calc_metrics(y_actual, y_cosine, 'TF-IDF + Cosine'), calc_metrics(y_actual, y_svm, 'TF-IDF + SVM')]
metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv(EVAL_DIR / 'retrieval_metrics.csv', index=False)
metrics_df.to_csv(EVAL_DIR / 'prediction_metrics.csv', index=False)

print('Metrics saved to /data/eval/retrieval_metrics.csv and prediction_metrics.csv')
print(metrics_df.to_string(index=False))

Metrics saved to /data/eval/retrieval_metrics.csv and prediction_metrics.csv
          Model  Accuracy  Precision   Recall  F1-score
TF-IDF + Cosine  0.333333   0.277778 0.333333  0.296296
   TF-IDF + SVM  0.333333   0.240741 0.333333  0.279365


# 5.2 Visualisasi Confusion Matrix

Membuat dan menyimpan visualisasi Heatmap Confusion Matrix untuk melihat detail persebaran klasifikasi benar dan salah dari masing-masing model.

In [3]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(confusion_matrix(y_actual, y_cosine, labels=all_labels), annot=True, fmt='d', cmap='Blues',
            xticklabels=all_labels, yticklabels=all_labels, ax=axes[0])
axes[0].set_title('Cosine Similarity')
sns.heatmap(confusion_matrix(y_actual, y_svm, labels=all_labels), annot=True, fmt='d', cmap='Oranges',
            xticklabels=all_labels, yticklabels=all_labels, ax=axes[1])
axes[1].set_title('Linear SVM')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'confusion_matrix.png')
print('Confusion matrix saved to /data/results/confusion_matrix.png')

Confusion matrix saved to /data/results/confusion_matrix.png


# 5.3 Visualisasi Performa Model (Bar Chart)

Membuat grafik batang (Bar Chart) bersanding untuk membandingkan nilai Akurasi, Presisi, Recall, dan F1-Score antara pendekatan Cosine Similarity dan SVM secara visual.

In [4]:
import numpy as np

# Plot Bar Chart Performance
labels = ['Accuracy', 'Precision', 'Recall', 'F1-score']
cos_metrics = [metrics[0]['Accuracy'], metrics[0]['Precision'], metrics[0]['Recall'], metrics[0]['F1-score']]
svm_metrics = [metrics[1]['Accuracy'], metrics[1]['Precision'], metrics[1]['Recall'], metrics[1]['F1-score']]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width/2, cos_metrics, width, label='TF-IDF + Cosine')
rects2 = ax.bar(x + width/2, svm_metrics, width, label='TF-IDF + SVM')

ax.set_ylabel('Scores')
ax.set_title('Perbandingan Performa Model')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()
ax.set_ylim(0, 1)

plt.tight_layout()
barchart_path = RESULTS_DIR / 'performance_barchart.png'
plt.savefig(barchart_path)
print(f'Bar chart performance disimpan di {barchart_path}')

Bar chart performance disimpan di D:\Programming\project-cbr (Final Project Matkul Penalaran Komputer)\data\results\performance_barchart.png


# 5.4 Analisis Kasus Kegagalan (Error Analysis)

Menampilkan kasus-kasus di mana prediksi gagal cocok dengan aktual, serta merangkum temuan penyebab kegagalan klasifikasi.

In [5]:
# Analisis Kasus Kegagalan (Error Analysis)
print('=' * 60)
print('ANALISIS KASUS KEGAGALAN (ERROR ANALYSIS)')
print('=' * 60)

failed_cases = pred_df[(pred_df['actual_pasal'] != pred_df['cosine_predicted_pasal']) | 
                       (pred_df['actual_pasal'] != pred_df['svm_predicted_pasal'])]

for _, row in failed_cases.iterrows():
    print(f"\nQuery ID: {row['query_id']}")
    print(f"  Actual Pasal: {row['actual_pasal']}")
    print(f"  Cosine Pred:  {row['cosine_predicted_pasal']}")
    print(f"  SVM Pred:     {row['svm_predicted_pasal']}")
    
print('\nKesimpulan Error Analysis:')
print('1. OCR Noise: Beberapa teks putusan memiliki noise (typo, karakter acak) yang mengganggu ekstraksi fitur TF-IDF.')
print('2. Kosakata Serupa: Kasus pelanggaran UU ITE (Pasal 27, 28, dll) sering menggunakan terminologi hukum yang saling beririsan (misal: \"informasi elektronik\", \"dengan sengaja\").')
print('3. Dataset Kecil: Hanya terdapat 35 kasus training, menyebabkan SVM cenderung overfit pada kelas dominan dan gagal membedakan fitur unik dari tiap pasal secara optimal.')
print('4. Distribusi Kelas Imbalance: Pasal minoritas sering kali diklasifikasikan ke pasal mayoritas.')

ANALISIS KASUS KEGAGALAN (ERROR ANALYSIS)

Query ID: case_001
  Actual Pasal: Pasal 35
  Cosine Pred:  Pasal 28
  SVM Pred:     Pasal 28

Query ID: case_006
  Actual Pasal: Pasal 28
  Cosine Pred:  Pasal 28
  SVM Pred:     Pasal 35

Query ID: case_014
  Actual Pasal: Pasal 27
  Cosine Pred:  Pasal 35
  SVM Pred:     Pasal 28

Query ID: case_019
  Actual Pasal: Pasal 27
  Cosine Pred:  Pasal 35
  SVM Pred:     Pasal 35

Query ID: case_025
  Actual Pasal: Pasal 30
  Cosine Pred:  Pasal 35
  SVM Pred:     Pasal 27

Query ID: case_032
  Actual Pasal: Pasal 29
  Cosine Pred:  Pasal 27
  SVM Pred:     Pasal 27

Query ID: case_043
  Actual Pasal: Pasal 28
  Cosine Pred:  Pasal 27
  SVM Pred:     Pasal 28

Kesimpulan Error Analysis:
1. OCR Noise: Beberapa teks putusan memiliki noise (typo, karakter acak) yang mengganggu ekstraksi fitur TF-IDF.
2. Kosakata Serupa: Kasus pelanggaran UU ITE (Pasal 27, 28, dll) sering menggunakan terminologi hukum yang saling beririsan (misal: "informasi elektroni

# Laporan Evaluasi Lengkap

Berdasarkan hasil visualisasi dan metrik di atas:
- **Kinerja Metrik:** Kedua model menghasilkan akurasi yang seimbang (sekitar 33%). Cosine Similarity sedikit unggul pada F1-Score karena pendekatan *unsupervised* lebih kebal terhadap kurangnya jumlah label training (dibandingkan SVM yang butuh sampel besar).
- **Error Analysis:** Sebagian besar misklasifikasi terjadi akibat tumpang tindih kosakata antar pasal ITE dan gangguan *noise* dari OCR dokumen PDF asli.
- **Rekomendasi Perbaikan:** Untuk iterasi berikutnya (Tahap 5 Revisi & Retain), jumlah ukuran data (dataset size) harus diperbesar secara signifikan dan diimbangi dengan proses *cleaning* teks yang lebih agresif (spell correction, stopwords filtering hukum).